In [1]:
using PowerModels, Ipopt, MathOptInterface, ProgressMeter, DelimitedFiles, Random, Statistics, DataFrames, CSV
const MOI = MathOptInterface
Random.seed!(42) 

function solve_DCOPF(case_path::String, num_samples::Int=50000, Variance::Float64=0.12)
    println("Samples: $(num_samples), Variance: $(Variance)")
    
    data = parse_file(case_path)
    base_loads = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"])

    load_bus_indices = sort(collect(keys(base_loads)))
    gen_indices = sort([g["index"] for (i, g) in data["gen"]])

    silent_optimizer = MOI.OptimizerWithAttributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")
    
    successful_samples = 0
    load_data_list = Vector{Vector{Float64}}()
    generation_data_list = Vector{Vector{Float64}}()
    cost_list = Float64[]

    total_loop_time = @elapsed begin
        @showprogress "Progress" for _ in 1:num_samples
            temp_data = deepcopy(data)
            new_loads = Dict{Int, Float64}()

            for bus_idx in load_bus_indices
                base_pd = base_loads[bus_idx]
                sigma = abs(base_pd * Variance)
                sampled_pd = max(0.0, base_pd + sigma * randn())
                new_loads[bus_idx] = sampled_pd
            end

            for (load_id, load_data) in temp_data["load"]
                bus_idx_int = load_data["load_bus"]
                if haskey(new_loads, bus_idx_int)
                    load_data["pd"] = new_loads[bus_idx_int]
                end
            end

            try
                result = solve_dc_opf(temp_data, silent_optimizer; setting = Dict("output" => Dict("branch_flows" => true), "conv.r_tol" => 1e-4))

                if result["termination_status"] == MOI.LOCALLY_SOLVED
                    push!(cost_list, result["objective"])
                    push!(load_data_list, [new_loads[i] for i in load_bus_indices])
                    push!(generation_data_list, [result["solution"]["gen"][string(i)]["pg"] for i in gen_indices])
                    successful_samples += 1
                end
            catch e
            end
        end 
    end 

    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")

    if !isempty(cost_list)
        avg_cost = mean(cost_list)
        println("Average cost: ", round(avg_cost, digits=4),"", string('$'), "/h")
    end

    if successful_samples > 0
        avg_solve_time_ms = (total_loop_time / successful_samples) * 1000 
        println("Average solving time for each sample: ", round(avg_solve_time_ms, digits=2), "ms")
    end

    output_dir = raw"C:\Users\Aloha\Desktop\dataset\DCOPF dataset\case30(v=0.12)" #change the output dir
    #output_dir = "/home/ubuntu/dcopf_case300(v=0.12)" 
    mkpath(output_dir)
    
    all_loads = hcat(load_data_list...)'
    all_generations = hcat(generation_data_list...)'
    case_name = split(basename(case_path), ".")[1]
    
    df_loads = DataFrame(all_loads, Symbol.("pd" .* string.(load_bus_indices)))
    CSV.write(joinpath(output_dir, "$(case_name)_loads.csv"), df_loads)
    df_gens = DataFrame(all_generations, Symbol.("gen" .* string.(gen_indices)))
    CSV.write(joinpath(output_dir, "$(case_name)_generations.csv"), df_gens)
    return all_loads, all_generations
end

# Main
case_file = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case30_ieee.m" #change the input path
solve_DCOPF(case_file, 11000, 0.12) #change the sample number and variance

Samples: 11000, Variance: 0.12
[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [1842.1527999999998, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 5: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 2: [5218.2254, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 6: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 3: Float64[]


Progress 100%|███████████████████████████████████████████| Time: 0:01:50


Successfully generated 10873 / 11000 samples in 111.05s.
Average cost: 7451.1326$/h
Average solving time for each sample: 10.21ms


([0.23752877986577575 0.02146600724365133 … 0.024739723668794865 0.10855576712748746; 0.24652674425863236 0.024834590575980156 … 0.027450123849060117 0.09036599156960304; … ; 0.2080560411181517 0.02761248165263378 … 0.02826629759149988 0.10222715798834046; 0.21458442208939402 0.022305347256976506 … 0.027262740848008653 0.10747960576553411], [2.1534840920984903 0.596846175809446 … 0.0 0.0; 2.1636902721678246 0.7003410759619921 … 0.0 0.0; … ; 2.166629671104486 0.6766310690082664 … 0.0 0.0; 2.1783900118643467 0.7228877792064659 … 0.0 0.0])

In [ ]:
#neww unify units
using PowerModels, Ipopt, MathOptInterface, ProgressMeter, DelimitedFiles, Random, Statistics, DataFrames, CSV
const MOI = MathOptInterface
Random.seed!(42)

function solve_DCOPF_pu(case_path::String, num_samples::Int=50000, Variance::Float64=0.12)
    println("Samples: $(num_samples), Variance: $(Variance)")
    
    data = parse_file(case_path)
    
    # --- 1. 构建 p.u. 参考 ---
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]
    baseMVA = ref[:baseMVA]

    # --- 2. 聚合 p.u. 基础负荷 ---
    # (这能正确处理单个母线上的多个负荷)
    bus_ids = sort(collect(keys(ref[:bus])))
    base_loads_pd_pu = Dict(id => 0.0 for id in bus_ids)
    for (_, load) in ref[:load]
        if get(load, "status", 1) == 1
            bus_id = load["load_bus"]
            base_loads_pd_pu[bus_id] += get(load, "pd", 0.0)
        end
    end
    
    # 只保留真正有负荷的母线
    load_bus_indices = sort([bus_id for (bus_id, pd) in base_loads_pd_pu if pd != 0.0])
    
    # 最终的 p.u. 基础负荷
    base_loads_pu = Dict(bus_id => base_loads_pd_pu[bus_id] for bus_id in load_bus_indices)
    
    gen_indices = sort(collect(keys(ref[:gen]))) # p.u. gen 索引

    silent_optimizer = MOI.OptimizerWithAttributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")
    
    successful_samples = 0
    load_data_list = Vector{Vector{Float64}}()
    generation_data_list = Vector{Vector{Float64}}()
    cost_list = Float64[]

    total_loop_time = @elapsed begin
        @showprogress "Progress" for _ in 1:num_samples
            
            load_scaling = Dict{Int, Float64}()
            new_loads_pu_agg = Dict{Int, Float64}() # 用于保存聚合的 p.u. 负荷

            # --- 3. 在 p.u. 系统中生成新负荷 ---
            for bus_idx in load_bus_indices
                base_pd_pu = base_loads_pu[bus_idx]
                sigma_pu = abs(base_pd_pu * Variance)
                sampled_pd_pu = max(0.0, base_pd_pu + sigma_pu * randn())
                
                new_loads_pu_agg[bus_idx] = sampled_pd_pu
                
                # 计算缩放系数，用于修改 MW 字典
                if base_pd_pu == 0.0
                    # 理论上不应发生，因为 load_bus_indices 已经过滤
                    load_scaling[bus_idx] = 0.0 
                else
                    load_scaling[bus_idx] = sampled_pd_pu / base_pd_pu
                end
            end

            # --- 4. 准备求解器输入 (必须是 MW) ---
            # 我们通过缩放原始 MW 数据来创建新的 MW 输入
            temp_data = deepcopy(data)
            for (load_id, load_data) in temp_data["load"]
                bus_idx_int = load_data["load_bus"]
                if haskey(load_scaling, bus_idx_int)
                    # 按 p.u. 缩放系数缩放原始 MW 值
                    load_data["pd"] = load_data["pd"] * load_scaling[bus_idx_int]
                    # (DCOPF 不关心 "qd", 但为了严谨性我们也缩放它)
                    if haskey(load_data, "qd")
                         load_data["qd"] = load_data["qd"] * load_scaling[bus_idx_int]
                    end
                end
            end

            try
                # --- 5. 求解 (输入 MW, 输出 p.u.) ---
                result = solve_dc_opf(temp_data, silent_optimizer; setting = Dict("output" => Dict("branch_flows" => true), "conv.r_tol" => 1e-4))

                if result["termination_status"] == MOI.LOCALLY_SOLVED
                    push!(cost_list, result["objective"])
                    
                    # --- 6. 保存 p.u. 值 ---
                    push!(load_data_list, [new_loads_pu_agg[i] for i in load_bus_indices])
                    push!(generation_data_list, [result["solution"]["gen"][string(i)]["pg"] for i in gen_indices])
                    
                    successful_samples += 1
                end
            catch e
            end
        end 
    end 

    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")

    if !isempty(cost_list)
        avg_cost = mean(cost_list)
        println("Average cost: ", round(avg_cost, digits=4),"", string('$'), "/h")
    end

    if successful_samples > 0
        avg_solve_time_ms = (total_loop_time / successful_samples) * 1000 
        println("Average solving time for each sample: ", round(avg_solve_time_ms, digits=2), "ms")
    end

    output_dir = raw"C:\Users\Aloha\Desktop\dataset\DCOPF dataset\case30(v=0.12)" #change the output dir
    mkpath(output_dir)
    
    all_loads = hcat(load_data_list...)'
    all_generations = hcat(generation_data_list...)'
    case_name = split(basename(case_path), ".")[1]
    
    # --- 7. 按 Python 脚本期望的格式保存 (pd{id}, pg{id}) ---
    df_loads = DataFrame(all_loads, Symbol.("pd" .* string.(load_bus_indices)))
    CSV.write(joinpath(output_dir, "$(case_name)_loads.csv"), df_loads)
    
    df_gens = DataFrame(all_generations, Symbol.("pg" .* string.(gen_indices)))
    CSV.write(joinpath(output_dir, "$(case_name)_generations.csv"), df_gens)
    
    return all_loads, all_generations
end

# Main
case_file = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case30_ieee.m" #change the input path
solve_DCOPF_pu(case_file, 50000, 0.12) #change the sample number and variance